# Анализ политического содержания комментариев

Анонимный анализ политических позиций, выраженных в текстах Telegram-каналов.

**Дисклеймер:** Результат описывает политическое содержание предоставленных сообщений и не является достоверным определением личных убеждений автора.

In [ ]:
# Cell 2: Конфигурация
from political_analysis.pipeline import PipelineConfig

CONFIG = PipelineConfig(
    database_path="/kaggle/input/telegram-political-comments/app.db",
    model_id="Qwen/Qwen3-4B-Instruct-2507",
    min_comments=20,
    min_comment_length=20,
    max_comments_per_user=300,
    batch_size=1,
    max_input_tokens=24000,
    max_new_tokens=1500,
    seed=42,
    resume=True,
    output_dir="/kaggle/working",
)

print(f"Model: {CONFIG.model_id}")
print(f"Min comments: {CONFIG.min_comments}")
print(f"Seed: {CONFIG.seed}")

In [ ]:
# Cell 2: Конфигурация
from political_analysis.pipeline import PipelineConfig

CONFIG = PipelineConfig(
    database_path="/kaggle/input/your-dataset/app.db",
    model_id="Qwen/Qwen3-4B-Instruct-2507",
    min_comments=20,
    min_comment_length=20,
    max_comments_per_user=300,
    batch_size=1,
    max_input_tokens=24000,
    max_new_tokens=1500,
    seed=42,
    resume=True,
    output_dir="/kaggle/working",
)

print(f"Model: {CONFIG.model_id}")
print(f"Min comments: {CONFIG.min_comments}")
print(f"Seed: {CONFIG.seed}")

In [ ]:
# Cell 3: Изучение структуры базы данных
from political_analysis.db_explorer import explore_database

structure = explore_database(CONFIG.database_path)
print(structure.summary())

In [ ]:
# Cell 4: Предварительная обработка
from political_analysis.anonymizer import Anonymizer
from political_analysis.preprocessor import load_user_comments

anonymizer = Anonymizer()
anonymizer.build_from_database(CONFIG.database_path, structure)
user_comments = load_user_comments(
    CONFIG.database_path, structure, anonymizer,
    min_comments=CONFIG.min_comments,
    min_comment_length=CONFIG.min_comment_length,
)

print(f"Users with sufficient comments: {len(user_comments)}")
for uc in user_comments[:5]:
    print(f"  {uc.anonymous_id}: {uc.political_comments} political comments")

In [ ]:
# Cell 5: Загрузка модели
from political_analysis.model_loader import load_model, ModelConfig

model_config = ModelConfig(
    model_id=CONFIG.model_id,
    max_new_tokens=CONFIG.max_new_tokens,
    seed=CONFIG.seed,
)
model, tokenizer = load_model(model_config)
print(f"Model loaded on: {model.device}")

In [ ]:
# Cell 6: Тестирование на одном пользователе
from political_analysis.preprocessor import limit_comments
from political_analysis.analyzer import run_inference, split_into_blocks, merge_block_results
from political_analysis.validator import validate_analysis_response

if user_comments:
    test_user = user_comments[0]
    test_user = limit_comments(test_user, CONFIG.max_comments_per_user, CONFIG.max_input_tokens)
    
    comments_for_model = [
        {"id": c.original_id, "text": c.text, "date": c.date, "channel": c.channel}
        for c in test_user.comments
    ]
    
    blocks = split_into_blocks(comments_for_model, 50)
    block_results = []
    
    for block in blocks:
        result = run_inference(model, tokenizer, block, CONFIG.max_new_tokens, seed=CONFIG.seed)
        if result.is_valid:
            print(f"Valid JSON received for block of {len(block)} comments")
            validated, errors = validate_analysis_response(result.parsed_json, [c["id"] for c in block])
            if errors:
                print(f"  Validation errors: {errors}")
            else:
                block_results.append(result.parsed_json)
        else:
            print(f"Invalid response: {result.error}")
    
    if block_results:
        merged = merge_block_results(block_results)
        print(f"\nMerged result for {test_user.anonymous_id}:")
        print(f"  Economic: {merged.get('content_analysis', {}).get('axes', {}).get('economic', {})}")
        print(f"  Confidence: {merged.get('overall_confidence')}")
else:
    print("No users with sufficient comments for testing")

In [ ]:
# Cell 7: Пакетная обработка
import logging
import time

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

from political_analysis.pipeline import Pipeline

pipeline = Pipeline(CONFIG)
pipeline.model = model
pipeline.tokenizer = tokenizer
pipeline.anonymizer = anonymizer

state = pipeline.run()

In [ ]:
# Cell 8: Экспорт результатов
from pathlib import Path
import json

output_dir = Path(CONFIG.output_dir)
print("Generated files:")
for f in sorted(output_dir.glob("*")):
    if f.is_file():
        print(f"  {f.name}: {f.stat().st_size:,} bytes")

In [ ]:
# Cell 9: Сводная статистика
import pandas as pd

summary_path = output_dir / "summary.csv"
if summary_path.exists():
    df = pd.read_csv(summary_path)
    print(f"Total users analyzed: {len(df)}")
    print(f"Average confidence: {df['overall_confidence'].mean():.2f}")
    print(f"Users with insufficient data: {df['insufficient_data'].sum()}")
    print(f"\nEconomic axis distribution:")
    print(f"  Mean left: {df['economic_left'].mean():.1f}")
    print(f"  Mean right: {df['economic_right'].mean():.1f}")
    print(f"\nTop ideologies:")
    for col in ["marxism", "anarchism", "liberalism"]:
        print(f"  {col}: {df[col].mean():.1f}")
    display(df.head(10))
else:
    print("No summary.csv found")